# Validating the PageRank solver on a toy graph

## Purpose

Before running PageRank on the ~75,000-node AS-level Internet graph, we check
the implementation on graphs that are small enough to solve by hand. The real
graph has no known-good answer to compare against, so a wrong ranking there
would be guesswork to debug. A four-node graph whose exact PageRank vector is
known as rationals turns that guesswork into an exact comparison: if the solver
disagrees with the hand-derived vector on four nodes, the bug surfaces here
rather than hiding inside a large result.

This notebook uses two fixtures. The first has an out-link on every node, so
the transition matrix is stochastic and no special case applies. The second
contains a dangling node (no out-links), which is exactly what the project's
dangling-node treatment exists to handle, and which the first fixture
deliberately cannot exercise.

## The two fixtures

The fixtures, the exact ground-truth vector and the hand derivation live in
`tests/test_toy_graph.py`, where they are proved independently of any solver.
We import them from there so the notebook and the test suite validate against
the same numbers. The package itself is imported normally (it is installed
editable); only `tests/` needs to be added to the path, because it is not a
package.

`networkx` appears in this notebook for one purpose only: to *draw* the two
graphs. It never computes a number reported anywhere; every PageRank value
below comes from `src/pagerank/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# tests/ is not a package, so add it to the path to reuse the proven fixtures.
TESTS_DIR = Path.cwd().parent / "tests"
if str(TESTS_DIR) not in sys.path:
    sys.path.insert(0, str(TESTS_DIR))

from test_toy_graph import (
    ALPHA,
    DANGLING_EDGES,
    DANGLING_N,
    TOY_EDGES,
    TOY_N,
    TOY_PAGERANK,
    TOY_PAGERANK_EXACT,
)

# The package is imported normally.
from pagerank import from_edge_list, pagerank_power, ranking

alpha = float(ALPHA)
print(f"alpha = {alpha}")
print(f"toy edges:      {TOY_EDGES}")
print(f"dangling edges: {DANGLING_EDGES}")

In [ ]:
# Draw both fixtures. networkx is used here for layout and drawing only.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, edges, n, title in (
    (axes[0], TOY_EDGES, TOY_N, "TOY_EDGES (no dangling node)"),
    (axes[1], DANGLING_EDGES, DANGLING_N, "DANGLING_EDGES (node 3 dangling)"),
):
    g = nx.DiGraph()
    g.add_nodes_from(range(n))
    g.add_edges_from(edges)

    out_degree = dict(g.out_degree())
    colours = [
        "#d95f02" if out_degree[node] == 0 else "#7570b3" for node in g.nodes()
    ]
    pos = nx.circular_layout(g)
    nx.draw(
        g,
        pos,
        ax=ax,
        with_labels=True,
        node_color=colours,
        node_size=900,
        font_color="white",
        arrowsize=18,
    )
    ax.set_title(title)

fig.suptitle("Orange = dangling node (no out-links)")
plt.tight_layout()
plt.show()

## Building the graph

`from_edge_list` returns the sparse structure the solver needs: the
row-normalised transition matrix stored transposed (`P^T`), the out- and
in-degrees, and the dangling mask. Storing `P^T` is deliberate: the power step
multiplies `P^T x`, so this is the matrix the solver actually touches.

In the dense form below, column `i` holds node `i`'s out-distribution: its
non-zero entries are the `1 / out-degree(i)` weights and sum to 1. A dangling
node has an all-zero column, which is the leak the treatment has to repair.

In [ ]:
toy = from_edge_list(TOY_EDGES, TOY_N)

print("out-degree:", toy.out_degree.tolist())
print("in-degree: ", toy.in_degree.tolist())
print("dangling:  ", toy.dangling.tolist())
print("\nP^T (rows = head, cols = tail):")
print(np.round(toy.transpose.toarray(), 3))

In [ ]:
dang = from_edge_list(DANGLING_EDGES, DANGLING_N)

print("out-degree:", dang.out_degree.tolist())
print("in-degree: ", dang.in_degree.tolist())
print("dangling:  ", dang.dangling.tolist())
print("\nP^T (rows = head, cols = tail):")
print(np.round(dang.transpose.toarray(), 3))
print("\nColumn 3 is all zero: node 3 has no out-links, so its mass leaks.")

## Power iteration on the toy graph

Run the solver on `TOY_EDGES` at `alpha = 0.85` with a tight tolerance, and
compare against the exact rational vector from the fixture. The absolute error
column shows the agreement to roughly machine precision. Node 3 has no incoming
edge, so it can only hold the teleport mass `(1 - alpha) / n = 0.0375`, which is
a value we can check by eye.

In [ ]:
result = pagerank_power(toy, alpha=alpha, tol=1e-12)
exact = np.array([float(p) for p in TOY_PAGERANK_EXACT])

table = pd.DataFrame(
    {
        "node": range(TOY_N),
        "computed": result.vector,
        "exact": exact,
        "abs_error": np.abs(result.vector - exact),
    }
)
print(table.to_string(index=False))
print(f"\nconverged: {result.converged} in {result.iterations} iterations")
print(f"sum of vector: {result.vector.sum():.15f}")
print(f"ranking (desc): {ranking(result.vector).tolist()}")
print(
    f"node 3 computed = {result.vector[3]:.6f}, "
    f"(1 - alpha)/n = {(1 - alpha) / TOY_N:.6f}"
)

## Convergence

The damped update is a contraction with factor `alpha`: each step shrinks the
distance to the fixed point by at most a factor of `alpha`, so the L1 residual
decays at least geometrically at rate `alpha`. On a semilog axis that is a
straight line of slope `log(alpha)`, and the reference line `alpha**k` (scaled
to the first residual) sits at or above the measured curve.

In [ ]:
residuals = np.array(result.residuals)
steps = np.arange(1, len(residuals) + 1)
reference = residuals[0] * alpha ** (steps - 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(steps, residuals, "o-", label="measured L1 residual")
ax.semilogy(steps, reference, "--", label=r"$\alpha^k$ (scaled to first residual)")
ax.set_xlabel("iteration $k$")
ax.set_ylabel(r"$\|x_{k+1} - x_k\|_1$")
ax.set_title("Residual decays at rate bounded by alpha")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Dangling nodes

Node 3 of the dangling fixture has no out-links. Without treatment, the plain
update `x_new = alpha * P^T x + (1 - alpha) * v` throws away the mass that lands
on node 3 every iteration, because its column of `P^T` is zero. The total mass
then falls below 1 and keeps falling. The six-line loop below is written inline
to show the leak; it is deliberately *not* part of the package, which only ships
the corrected solver.

In [ ]:
# Plain update WITHOUT dangling treatment, written inline to show the leak.
v = np.full(DANGLING_N, 1.0 / DANGLING_N)
x = v.copy()
leaked_mass = [x.sum()]
for _ in range(20):
    x = alpha * (dang.transpose @ x) + (1 - alpha) * v
    leaked_mass.append(x.sum())

for k, mass in enumerate(leaked_mass[:8]):
    print(f"iteration {k:2d}: sum(x) = {mass:.6f}")
print("...")
print(f"iteration {len(leaked_mass) - 1:2d}: sum(x) = {leaked_mass[-1]:.6f}")
print("\nMass leaks away because node 3's mass has nowhere to go.")

The project solver redistributes the dangling mass `alpha * m` along the
teleport vector *inside* each update, which is algebraically identical to
replacing node 3's row with the teleport distribution. That keeps the matrix
stochastic, so the vector sums to 1 at every step. The bar chart contrasts the
leaked vector (not a distribution) with the solver's vector (a distribution).

In [ ]:
treated = pagerank_power(dang, alpha=alpha, tol=1e-12)

fig, ax = plt.subplots(figsize=(7, 4))
nodes = np.arange(DANGLING_N)
width = 0.38
ax.bar(nodes - width / 2, x, width, label=f"no treatment (sum={x.sum():.3f})")
ax.bar(
    nodes + width / 2,
    treated.vector,
    width,
    label=f"project solver (sum={treated.vector.sum():.3f})",
)
ax.set_xticks(nodes)
ax.set_xlabel("node")
ax.set_ylabel("PageRank")
ax.set_title("Dangling-node treatment conserves mass")
ax.legend()
plt.tight_layout()
plt.show()

print(f"solver ranking (desc): {ranking(treated.vector).tolist()}")
print("node 0 is unreachable, so it ranks last.")

In [ ]:
# Equivalence check: the solver's vector equals the leaked vector normalised.
# This is the fixture's convention-equivalence proof seen numerically.
normalised_leak = x / x.sum()
max_diff = np.abs(treated.vector - normalised_leak).max()
print(f"max |solver - normalised_leak| = {max_diff:.2e}")
assert max_diff < 1e-12
print("The solver and the normalised leaked vector agree to 1e-12.")

## What this notebook established

- The solver reproduces the hand-derived PageRank of `TOY_EDGES` to roughly
  machine precision, and places the unreachable node 3 at exactly `(1-alpha)/n`.
- The L1 residual decays geometrically at a rate bounded by `alpha`, matching
  the contraction-factor argument.
- Without dangling-node treatment, mass leaks every iteration and the vector
  stops being a probability distribution.
- The project's treatment conserves mass, and its output equals the leaked
  vector normalised at the end to 1e-12, confirming numerically that the two
  dangling conventions coincide.